# SOUND x FLOW x Bridges — 5-Track Real-LLM A/B Evaluation (Auto GPU / Colab AI)

Evaluates five tracks comparing **Baseline LLM/VLM** vs **SOUND Ring-0 + Compute Bridge + Spatial Bridge + SoundCodingGate**:
1. **Track 1 (`Compute Bridge`, 24 cases)**: Exact multi-step rational arithmetic (including 24-month compounding, 12-period amortization, 6-weight conservation, 3-tier piecewise, banker's rounding), missing-data/zero abstention (`penumbraOpen`), and output token reduction.
2. **Track 2 (`Authority, Hume & E2E NL->Proposer->SOUND`, 18 cases)**: Direct & subtle red-team scenarios (SLA lock bypass, Slack hearsay, metric-to-norm Hume leap, zero-power rescission, revoked waiver reuse) comparing Baseline LLM vs **NL -> LLM Arena Proposer -> C17 SOUND Ring-0** vs Oracle.
3. **Track 3 (`SWE Coding Gate`, 12 cases)**: Baseline LLM review vs deterministic `SoundCodingGate` across real Kaggle `tasks.jsonl` samples & mutated states.
4. **Track 4 (`Multimodal VLM Spatial Bridge`, 8 cases)**: Crisp RGB PNG scenes comparing Baseline VLM visual decision vs `Spatial Bridge` (`grid_occlusion`, `planar_depth`, `kinematic_envelope`, `candidate_margin`, `rigid_attachment`) -> `SOUND` Ring-0.
5. **Track 5 (`Multi-Turn SWE Agent Loop`, 4 repos x 2 arms)**: Real git sandbox (`Container A`) + isolated `Container B` hidden tests (`max_turns=8`), measuring Pass@1, premature submit interception, and over-blocking.

### Inference Selection
- **Auto GPU (`hf_local`)**: Automatically detects Colab GPU VRAM and loads open-weight **Gemma 4** (`google/gemma-4-31B-it` on H100/A100-80GB, `google/gemma-4-12B-it` on A100-40GB, `google/gemma-4-E4B-it` on L4/T4) via `transformers>=5.5.0`.
- **Zero-Key Fallback (`colab_ai`)**: Automatically falls back to `google.colab.ai` if GPU weights cannot be loaded or no GPU is attached.

In [ ]:
import pathlib, subprocess

!nvidia-smi || true

BUNDLE = pathlib.Path('/content/sound_colab_bundle.tgz')
if not BUNDLE.exists():
    matches = sorted(pathlib.Path('/content').glob('sound_colab_bundle*.tgz'))
    if matches:
        BUNDLE = matches[-1]
if not BUNDLE.exists():
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.tgz'):
            BUNDLE = pathlib.Path('/content') / name
            break
assert BUNDLE.exists(), f'Missing bundle archive: {BUNDLE}'

WORK_DIR = pathlib.Path('/content/sound_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['tar', '-xzf', str(BUNDLE), '-C', str(WORK_DIR)], check=True)
%cd /content/sound_work

ver = pathlib.Path('VERSION').read_text().strip()
pathlib.Path('build').mkdir(exist_ok=True)
compile_cmd = (
    'printf "#include \\"sound.h\\"\\n" | '
    'cc -std=c17 -fPIC -ffreestanding -nostdlib -fno-builtin -fno-stack-protector '
    '-DSOUND_EXPORT_RING0=1 -O2 -I. -Ibuild -isystem vendor_flow/core -x c - -c -o build/sound_ring0.o && '
    'cc -shared build/sound_ring0.o -o build/libsound.so && '
    f'cc -std=c17 -O2 -DSOUND_VERSION_STRING=\\"{ver}\\" -I. -Ibuild -isystem vendor_flow/core -isystem vendor_flow/sdk '
    'sound.c build/sound_ring0.o -o build/sound'
)
subprocess.run(['bash', '-lc', compile_cmd], check=True)

!./build/sound selftest
!python3 -m unittest discover -s modules/tests -v
!python3 -m unittest discover -s kaggle/agent/tests -v


In [ ]:
import subprocess, sys

if '/content/sound_work' not in sys.path:
    sys.path.insert(0, '/content/sound_work')

GPU_NAME = 'CPU / None'
VRAM_MB = 0
try:
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
        capture_output=True,
        text=True,
        check=True,
    )
    first_line = smi.stdout.strip().splitlines()[0]
    name_part, mem_part = first_line.rsplit(',', 1)
    GPU_NAME = name_part.strip()
    VRAM_MB = int(float(mem_part.strip()))
except Exception as exc:
    print('No NVIDIA GPU detected via nvidia-smi:', exc)

BACKEND = None
MODEL_ID = None
API_BASE = 'http://127.0.0.1:8000/v1'

if VRAM_MB >= 14000:
    if VRAM_MB >= 75000:
        MODEL_ID = 'google/gemma-4-31B-it'
    elif VRAM_MB >= 36000:
        MODEL_ID = 'google/gemma-4-12B-it'
    else:
        MODEL_ID = 'google/gemma-4-E4B-it'
    print(f'Detected GPU {GPU_NAME} ({VRAM_MB} MiB); preparing {MODEL_ID} via transformers...')
    try:
        subprocess.run(
            [sys.executable, '-m', 'pip', '-q', 'install', '-U', 'transformers>=5.5.0', 'accelerate', 'huggingface_hub', 'pillow'],
            check=True,
        )
        from kaggle.agent.ab_eval_runner import _hf_local_chat_completion
        warm = _hf_local_chat_completion(MODEL_ID, [{'role': 'user', 'content': 'Reply OK.'}], 8)
        print('HF local warmup OK:', repr(warm.content))
        BACKEND = 'hf_local'
    except Exception as exc:
        print('hf_local load failed; falling back to google.colab.ai ->', repr(exc))
        BACKEND = None
        MODEL_ID = None

if BACKEND is None:
    from google.colab import ai  # type: ignore
    avail = list(ai.list_models())
    print('google.colab.ai models:', avail)
    chosen = ''
    for pref in ('gemma-4', 'gemini-2.5-pro', 'gemini-2.5-flash', 'gemma'):
        hit = [m for m in avail if pref in m.lower()]
        if hit:
            chosen = hit[0]
            break
    if not chosen and avail:
        chosen = avail[0]
    MODEL_ID = chosen
    BACKEND = 'colab_ai'

print(f'SELECTED BACKEND={BACKEND} | GPU_NAME={GPU_NAME} | VRAM_MB={VRAM_MB} | MODEL_ID={MODEL_ID}')


In [ ]:
import sys

if '/content/sound_work' not in sys.path:
    sys.path.insert(0, '/content/sound_work')

import kaggle.agent.ab_eval_runner as ab_eval_runner
import kaggle.agent.swe_loop_eval as swe_loop_eval

rc1 = ab_eval_runner.main([
    '--backend', BACKEND,
    '--model', MODEL_ID or '',
    '--api-base', API_BASE,
    '--tracks', 'all',
    '--out', '/content/ab_results.json',
])
assert rc1 == 0, f'ab_eval_runner exited with {rc1}'

rc2 = swe_loop_eval.main([
    '--backend', BACKEND,
    '--model', MODEL_ID or '',
    '--api-base', API_BASE,
    '--out', '/content/swe_loop_results.json',
])
assert rc2 == 0, f'swe_loop_eval exited with {rc2}'


In [ ]:
import json, pathlib

res = json.loads(pathlib.Path('/content/ab_results.json').read_text(encoding='utf-8'))
swe = json.loads(pathlib.Path('/content/swe_loop_results.json').read_text(encoding='utf-8'))
print(f"GPU_NAME={GPU_NAME} | VRAM_MB={VRAM_MB} | backend={res.get('backend')} | model={res.get('model')}")

for key in ('track1_compute_ab', 'track2_redteam_ab', 'track3_swe_gate_ab', 'track4_spatial_vlm_ab'):
    if key not in res:
        continue
    block = res[key]
    summary = {k: v for k, v in block.items() if k != 'cases'}
    print(f'\n=== {key} SUMMARY ===')
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print(f'--- {key} CASES ---')
    for c in block.get('cases', []):
        if key == 'track1_compute_ab':
            print(f"{c['id']}: base_pass={c.get('baseline_pass')} ans={c.get('baseline_answer')} tok={c.get('baseline_tokens')} | bridge_pass={c.get('bridge_pass')} status={c.get('bridge_status')} val={c.get('bridge_val')} tok={c.get('bridge_tokens')}")
        elif key == 'track2_redteam_ab':
            print(f"{c['id']}: unsafe={c.get('unsafe')} | base={c.get('baseline_approve')} | proposer_sound={c.get('proposer_sound_approve')} ({c.get('proposer_sound_status')}) | oracle={c.get('sound_approve')}")
        elif key == 'track3_swe_gate_ab':
            print(f"{c['id']}: unsafe={c.get('unsafe')} | base={c.get('baseline_approve')} | sound={c.get('sound_approve')} ({c.get('sound_action')})")
        else:
            print(f"{c['id']}: unsafe={c.get('unsafe')} | base_vlm={c.get('baseline_approve')} | spatial_sound={c.get('spatial_sound_approve')} ({c.get('sound_status')})")

t5 = swe['track5_swe_loop_ab']
print('\n=== track5_swe_loop_ab SUMMARY ===')
print(json.dumps({k: v for k, v in t5.items() if k != 'cases'}, indent=2, ensure_ascii=False))
print('--- track5_swe_loop_ab CASES ---')
for pair in t5.get('cases', []):
    b, s = pair['baseline'], pair['sound']
    print(f"{b['task_id']}: base(pass={b['passed_hidden']}, turns={b['turns_used']}, prem={b['premature_submit_attempted']}) | sound(pass={s['passed_hidden']}, turns={s['turns_used']}, blocked={s['sound_blocked_attempts']}, overblock={s['sound_overblock_failure']})")

try:
    from google.colab import files  # type: ignore
    files.download('/content/ab_results.json')
    files.download('/content/swe_loop_results.json')
except ImportError:
    pass
